In [1]:
import numpy as np
from scipy.linalg import eigh
import scipy
from scipy.optimize import minimize

In [2]:
m_e  = 1
m_star = 0.16
eps_0 = 1/(4*np.pi)
eps_r = 8.8
eps = eps_r * eps_0
alpha_r_znse = ((m_e*m_star)/eps_r)**2

In [3]:
# alpha_dict = {
#     "a_1": 8.021420155 * alpha_r_znse,
#     "a_2": 1.467821061 * alpha_r_znse,
#     "a_3": 0.4077767635 * alpha_r_znse,
#     "a_4": 0.1353374420 * alpha_r_znse
# } 

alpha_dict = {
    "a_1": 17.38354739 * alpha_r_znse,
    "a_2": 3.185489246 * alpha_r_znse,
    "a_3": 0.8897299079 * alpha_r_znse,
    "a_4": 0.3037874103 * alpha_r_znse,
    "a_5": 0.1144784984 * alpha_r_znse
} # ST0-5G

In [4]:
N = len(alpha_dict)

In [5]:
def pe_H_func(n, empty_dict, dict_of_alpha):
 
    constant_pe = -m_e**2 / (2 * eps)

    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "V_H_" + str(i) + str(j)
            denominator = (dict_of_alpha['a_'+str(i)]) + (dict_of_alpha['a_'+str(j)])
            value = 1/denominator
            empty_dict[key] = value * constant_pe
    return empty_dict

In [6]:
def pe_cc_func(n, empty_dict, dict_of_alpha, r_cc):
 
    constant = -(m_e**2) * ((1/eps_0) - (1/eps))



    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "V_cc_" + str(i) + str(j)
            p =  (dict_of_alpha['a_'+str(i)]) + (dict_of_alpha['a_'+str(j)])
            w1 = 0.85355339
            t1 = 2 - (2**0.5)
            w2 = 0.14644661
            t2 = 2 + (2**0.5)
            w1t1 = (r_cc**2)*w1*np.exp(-p*(r_cc**2)*(t1**2))*t1
            w2t2 = (r_cc**2)*w2*np.exp(-p*(r_cc**2)*(t2**2))*t2
            
            value = w1t1 + w2t2 
            
            empty_dict[key] = value * constant
    return empty_dict

In [7]:
def ke_func(n, empty_dict, dict_of_alpha):
    hbar = 1  
    constant_ke = (3*(hbar**2))/(m_star)

    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "T_" + str(i) + str(j)
            numerator = (dict_of_alpha['a_'+str(i)])*(dict_of_alpha['a_'+str(j)])* (np.pi)**1.5
            denominator = ((dict_of_alpha['a_'+str(i)]) + (dict_of_alpha['a_'+str(j)]))**2.5
            value = numerator/denominator
            empty_dict[key] = value * constant_ke
    return empty_dict

In [8]:
def overlap_func(n, empty_dict, dict_of_alpha):

    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "S_" + str(i) + str(j)
            denominator = ((dict_of_alpha['a_'+str(i)]) + (dict_of_alpha['a_'+str(j)]))
            value = (np.pi/denominator)**1.5
            empty_dict[key] = value
    return empty_dict

In [9]:
def total_pe_construction(n, empty_dict,pe_H_dictionary, pe_cc_dictionary):
    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "V_" + str(i) + str(j)
            value = pe_H_dictionary["V_H_"+str(i)+str(j)]+ pe_cc_dictionary["V_cc_"+str(i)+str(j)]
            empty_dict[key] = value
    return empty_dict

In [10]:
def Hamilt_construction(H,n,ke_dictionary, pe_dictionary):
    H = np.zeros((n,n))
    for i in range(0,n):
        for j in range(0,n):
            H[i][j] = ke_dictionary['T_'+str(i+1)+str(j+1)] + pe_dictionary['V_'+str(i+1)+str(j+1)]
    
    return H

In [11]:
def overlap_matrix_construction(S,n, overlap_dict):
    S = np.zeros((n,n))
    for i in range(0,n):
        for j in range(0,n):
            S[i][j] =  overlap_dict['S_'+str(i+1)+str(j+1)]
    
    return S

In [12]:
S_dict = {}
S_dict = overlap_func(N,S_dict,alpha_dict)
T_dict = {}
T_dict = ke_func(N,T_dict,alpha_dict)
V_H_dict = {}
V_H_dict = pe_H_func(N,V_H_dict,alpha_dict)
S = ''
S = overlap_matrix_construction(S,N,S_dict)

In [13]:
def objective_function(r_cc, target_value):
    r_cc = r_cc[0]
  
    V_cc_dict = {}
    V_cc_dict = pe_cc_func(N,V_cc_dict,alpha_dict, r_cc)
    V_dict = {}
    V_dict = total_pe_construction(N,V_dict,V_H_dict,V_cc_dict)
    H = ''
    H = Hamilt_construction(H,N,T_dict,V_dict)
    E_vals, E_vectors = eigh(H, S)
    min_E =  (E_vals[0] * 27211.38)


    current_value = min_E
    squared_error = (current_value - target_value) ** 2
    return squared_error

In [14]:


target = -28.1
initial_guess = [0.3542]  # Must be a list or array

# 4. Run the optimizer
result = minimize(objective_function, initial_guess, args=(target,))


In [16]:
result.x

array([0.15214322])